# Demonstration
In the code below, efficiency gains from the tools discussed in this chapter are demonstrated. Three tools are evaluated, specifically

1. `numpy.ix_` for efficient matrix assembly (vs. double loops)
2. `scipy.sparse` for efficient matrix storage (vs. dense matrices)
3. `scipy.linalg.solve` for solving systems (vs. matrix inversion)

The example is based on the 2D heat equation but with a simplified finite element implementation, the element matrix is hard-coded rather than computed from finite element equations. 


In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import time
from scipy.linalg import solve as dense_solve

# Set up problem parameters
print("=" * 60)
print("2D HEAT CONDUCTION - FINITE ELEMENT DEMO")
print("=" * 60)

# Problem setup
nx, ny = 20, 20  # Grid points in x and y directions
n_nodes = nx * ny
n_elements = (nx-1) * (ny-1) * 2  # 2 triangular elements per quad

print(f"Problem size: {n_nodes} nodes, {n_elements} elements")
print(f"System matrix size: {n_nodes} x {n_nodes}")
print()

# Generate mesh coordinates
x = np.linspace(0, 1, nx)
y = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x, y)
coords = np.column_stack([X.ravel(), Y.ravel()])

# Generate element connectivity (triangular elements)
def generate_connectivity():
    """Generate element connectivity for triangular mesh"""
    elements = []
    for i in range(nx-1):
        for j in range(ny-1):
            # Node indices for current quad
            n1 = i * ny + j
            n2 = (i+1) * ny + j
            n3 = (i+1) * ny + (j+1)
            n4 = i * ny + (j+1)
            
            # Two triangular elements per quad
            elements.append([n1, n2, n3])  # Lower triangle
            elements.append([n1, n3, n4])  # Upper triangle
    
    return np.array(elements)

elements = generate_connectivity()

print("=" * 60)
print("1. MATRIX ASSEMBLY: numpy.ix_ vs. Double Loops")
print("=" * 60)

# Element stiffness matrix for 2D heat conduction (simplified)
def element_stiffness_matrix():
    """Simple 3x3 element stiffness matrix for demonstration"""
    return np.array([
        [2.0, -1.0, -1.0],
        [-1.0, 2.0, -1.0],
        [-1.0, -1.0, 2.0]
    ])

K_elem = element_stiffness_matrix()

# METHOD 1: Naive approach with double loops
print("Assembly method 1: Double loop assembly (SLOW)")
start_time = time.time()

K_dense_naive = np.zeros((n_nodes, n_nodes))
for elem_idx, elem_nodes in enumerate(elements):
    for i in range(3):
        for j in range(3):
            K_dense_naive[elem_nodes[i], elem_nodes[j]] += K_elem[i, j]

time_naive = time.time() - start_time
print(f"Time taken: {time_naive:.4f} seconds")

# METHOD 2: Efficient approach using numpy.ix_
print("\nAssembly method 2: numpy.ix_ assembly (FAST)")
start_time = time.time()

K_dense_efficient = np.zeros((n_nodes, n_nodes))
for elem_nodes in elements:
    # numpy.ix_ creates index arrays for efficient matrix indexing
    ix = np.ix_(elem_nodes, elem_nodes)
    K_dense_efficient[ix] += K_elem

time_efficient = time.time() - start_time
print(f"Time taken: {time_efficient:.4f} seconds")
print(f"Speedup: {time_naive/time_efficient:.2f}x")

# Verify both methods give same result
print(f"Methods produce identical results: {np.allclose(K_dense_naive, K_dense_efficient)}")

print("\n" + "=" * 60)
print("2. MATRIX STORAGE: scipy.sparse vs. Dense Matrices")
print("=" * 60)

# METHOD 3: Sparse matrix assembly (most efficient for large systems)
print("Assembly method 3: Sparse matrix assembly (BEST for large systems)")
start_time = time.time()

# Pre-allocate arrays for sparse matrix construction
row_indices = []
col_indices = []
data = []

for elem_nodes in elements:
    for i in range(3):
        for j in range(3):
            row_indices.append(elem_nodes[i])
            col_indices.append(elem_nodes[j])
            data.append(K_elem[i, j])

# Create sparse matrix using COO format, then convert to CSR
K_sparse = sp.csr_array((data, (row_indices, col_indices)), 
                        shape=(n_nodes, n_nodes))

time_sparse = time.time() - start_time
print(f"Time taken: {time_sparse:.4f} seconds")

# Memory usage comparison
dense_memory = K_dense_efficient.nbytes / (1024**2)  # MB
sparse_memory = (K_sparse.data.nbytes + K_sparse.indices.nbytes + 
                K_sparse.indptr.nbytes) / (1024**2)  # MB

print(f"\nMemory usage comparison:")
print(f"Dense matrix: {dense_memory:.2f} MB")
print(f"Sparse matrix: {sparse_memory:.2f} MB")
print(f"Memory savings: {(1 - sparse_memory/dense_memory)*100:.1f}%")

# Sparsity pattern
nnz = K_sparse.nnz
total_entries = n_nodes * n_nodes
sparsity = (1 - nnz/total_entries) * 100
print(f"Matrix sparsity: {sparsity:.1f}% (only {nnz} non-zero entries out of {total_entries})")

print("\n" + "=" * 60)
print("3. SOLVING SYSTEMS: scipy.linalg.solve vs. Matrix Inversion")
print("=" * 60)

# Create a realistic right-hand side (heat source)
rhs = np.ones(n_nodes)

# Apply boundary conditions (zero temperature at boundaries)
boundary_nodes = []
for i in range(nx):
    boundary_nodes.extend([i*ny, i*ny + ny-1])  # Top and bottom edges
for j in range(1, ny-1):
    boundary_nodes.extend([j, (nx-1)*ny + j])   # Left and right edges

boundary_nodes = list(set(boundary_nodes))
rhs[boundary_nodes] = 0

# Add identity rows for boundary conditions
K_sparse_bc = K_sparse.copy()
for node in boundary_nodes:
    start, end = K_sparse_bc.indptr[node], K_sparse_bc.indptr[node+1]
    K_sparse_bc.data[start:end] = 0
    K_sparse_bc[node, node] = 1

print("Right-hand side (heat source) created with boundary conditions applied")

# METHOD 1: Matrix inversion approach)
print("\nSolving method 1: Matrix inversion (NEVER DO THIS ON LARGE SYSTEMS!)")

# For demo purposes, solve a smaller system to show inversion
if n_nodes < 500:
    n_small = min(500, n_nodes)  # Limit size for inversion demo
    K_small = K_sparse_bc[:n_small, :n_small].toarray()
    rhs_small = rhs[:n_small]

    start_time = time.time()
    try:
        K_inv = np.linalg.inv(K_small)
        solution_inv = K_inv @ rhs_small
        time_inversion = time.time() - start_time
        print(f"Time taken (inversion): {time_inversion:.4f} seconds")
    except np.linalg.LinAlgError:
        print("Matrix inversion failed (singular matrix)")
        time_inversion = float('inf')
else:
    print("Matrix inversion skipped (matrix too large)")

# METHOD 2: Direct solve with scipy.linalg.solve (GOOD for dense)
print(f"\nSolve method 2: scipy.linalg.solve (dense)")
start_time = time.time()
solution_dense = dense_solve(K_small, rhs_small)
time_dense_solve = time.time() - start_time
print(f"Time taken (dense solve): {time_dense_solve:.4f} seconds")

# METHOD 3: Sparse solver (BEST for large sparse systems)
print(f"\nSolve method 3: scipy.sparse.linalg.spsolve (sparse - BEST)")
start_time = time.time()
solution_sparse = spla.spsolve(K_sparse_bc, rhs)
time_sparse_solve = time.time() - start_time
print(f"Time taken (sparse solve): {time_sparse_solve:.4f} seconds")

# Compare solutions
if time_inversion != float('inf'):
    print(f"\nSolution comparison (first {n_small} nodes):")
    print(f"Inversion vs Dense solve: {np.allclose(solution_inv, solution_dense)}")
    print(f"Dense vs Sparse solve: {np.allclose(solution_dense, solution_sparse[:n_small])}")

print(f"\nPerformance summary:")
if time_inversion != float('inf'):
    print(f"Inversion speedup over solve: {time_inversion/time_dense_solve:.2f}x SLOWER")
print(f"Sparse solver speedup: {time_dense_solve/time_sparse_solve:.2f}x faster than dense")

print("\n" + "=" * 60)
print("4. VISUALIZATION AND SUMMARY")
print("=" * 60)

# Reshape solution for visualization
solution_2d = solution_sparse.reshape((nx, ny))

# Create visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Temperature distribution
im1 = ax1.contourf(X, Y, solution_2d, levels=20, cmap='hot')
ax1.set_title('Temperature Distribution')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
plt.colorbar(im1, ax=ax1)

# Plot 2: Sparsity pattern
ax2.spy(K_sparse, markersize=0.5)
ax2.set_title(f'Matrix Sparsity Pattern\n({sparsity:.1f}% sparse)')

# Plot 3: Assembly time comparison
methods = ['Double Loop', 'numpy.ix_', 'Sparse Assembly']
times = [time_naive, time_efficient, time_sparse]
colors = ['red', 'orange', 'green']
bars = ax3.bar(methods, times, color=colors)
ax3.set_ylabel('Time (seconds)')
ax3.set_title('Assembly Time Comparison')
ax3.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{time_val:.4f}s', ha='center', va='bottom')

# Plot 4: Memory usage comparison
storage_types = ['Dense', 'Sparse']
memory_usage = [dense_memory, sparse_memory]
colors = ['red', 'green']
bars = ax4.bar(storage_types, memory_usage, color=colors)
ax4.set_ylabel('Memory Usage (MB)')
ax4.set_title('Memory Usage Comparison')

# Add value labels on bars
for bar, mem_val in zip(bars, memory_usage):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{mem_val:.2f}MB', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nProblem solved: {n_nodes} nodes, {n_elements} elements")
print(f"Total runtime: {time.time() - start_time:.2f} seconds")
